In [57]:
from sqlalchemy import create_engine
import pandas as pd

sqlite_db_path = r"c:\Users\m\Downloads\test.sqlite"

sqlite_db_connstring = f"sqlite:///{sqlite_db_path}"

ENGINE = create_engine(sqlite_db_connstring)

multiqc_data = '''
SELECT s.sample_name, sdt.data_key, NULLIF(sd.value, 'None') AS value, sd.report_id, s.sample_id 
            FROM sample_data sd
            JOIN sample_data_type sdt ON sdt.sample_data_type_id = sd.sample_data_type_id
            JOIN sample s ON sd.sample_id = s.sample_id
            WHERE sdt.data_section != 'general_stats';
        '''

plot_data = '''
select s.sample_name, s.sample_id, pd."data", pd.config_id, pd.report_id, pc.config_name, pc.config_dataset
from plot_data pd 
inner join sample s using (sample_id)
inner join plot_config pc using (config_id)
where pc.config_type = 'xy_line' and pc.config_dataset != 'Estimated Average Coverage';
'''
multiqc_data = pd.read_sql(multiqc_data, ENGINE)
plot_data = pd.read_sql(plot_data, ENGINE)

In [89]:
plot_data['plot'] = plot_data['config_name'] + '_' + plot_data['config_dataset']

before = set(plot_data['sample_name'])
plot_piv = plot_data.pivot(index='sample_name', columns='plot', values='data')
plot_piv.columns.name = None
plot_piv = plot_piv.reset_index()
after = set(plot_piv['sample_name'])

assert before == after
assert len(plot_piv['sample_name']) == len(set(plot_piv['sample_name']))

In [90]:
before = set(multiqc_data['sample_name'])
multiqc_piv = multiqc_data.pivot(index='sample_name', columns='data_key', values='value')
multiqc_piv.columns.name = None

multiqc_piv = multiqc_piv.reset_index()

after = set(multiqc_piv['sample_name'])

assert len(multiqc_piv['sample_name']) == len(set(multiqc_piv['sample_name']))
assert before == after

In [144]:
multiqc_plus = multiqc_piv.merge(plot_piv, on='sample_name', how='outer')
multiqc_plus['library_id'] = multiqc_plus['sample_name'].apply(lambda x: x.split('_')[1])
multiqc_plus['lane'] = multiqc_plus['sample_name'].apply(lambda x: x.split('_')[2])